## 1. Setup

In [5]:
# Cell 1
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

price_panel = pd.read_csv(PROCESSED_DIR / "price_panel.csv", index_col="date")
price_panel.index = pd.to_datetime(price_panel.index) 

print(f"Price panel loaded: {price_panel.shape[0]} dates x {price_panel.shape[1]} tickers")
print(f"Date range: {price_panel.index.min().date()} to {price_panel.index.max().date()}")
price_panel.head()

Price panel loaded: 6511 dates x 609 tickers
Date range: 1990-01-02 to 2015-10-30


,A,AAL,AAP,AAPL,ABBV,ABT,ACGL,ACN,ACV,ADBE,...,XEL,XOM,XRAY,XRX,XYL,YUM,ZBH,ZBRA,ZION,ZTS
date,,,,,,,,,,,,,,,,,,,,,
1990-01-02,NaN,NaN,NaN,0.260241,NaN,1.606693,NaN,NaN,NaN,1.188339,...,3.773864,3.656284,0.788891,7.544384,NaN,NaN,NaN,NaN,1.640894,NaN
1990-01-03,NaN,NaN,NaN,0.261987,NaN,1.612493,NaN,NaN,NaN,1.247023,...,3.762215,3.619722,0.788891,7.609700,NaN,NaN,NaN,NaN,1.670196,NaN
1990-01-04,NaN,NaN,NaN,0.262861,NaN,1.609593,NaN,NaN,NaN,1.305707,...,3.669033,3.583157,0.757335,7.626031,NaN,NaN,NaN,NaN,1.670196,NaN
1990-01-05,NaN,NaN,NaN,0.263735,NaN,1.592192,NaN,NaN,NaN,1.335049,...,3.634091,3.564878,0.804668,7.642362,NaN,NaN,NaN,NaN,1.670196,NaN
1990-01-08,NaN,NaN,NaN,0.265481,NaN,1.600355,NaN,NaN,NaN,1.352693,...,3.599148,3.619722,0.773112,7.593371,NaN,NaN,NaN,NaN,1.670196,NaN


## 2. Daily returns and binary target

In [3]:
# Cell 2
daily_returns = price_panel.pct_change()

def compute_median_split_target(returns: pd.DataFrame) -> pd.DataFrame:
    """Binary target: 1 if a stock's NEXT-day return >= that day's
    cross-sectional median return, else 0. NaN where the next-day return
    is unavailable (e.g. end of series, or stock not yet listed)."""
    next_day_returns = returns.shift(-1)
    daily_median = next_day_returns.median(axis=1)
    target = next_day_returns.ge(daily_median, axis=0).astype(float)
    target[next_day_returns.isna()] = np.nan
    return target

targets = compute_median_split_target(daily_returns)

print(f"Daily returns: {daily_returns.shape}")
print(f"Targets:       {targets.shape}")
print(f"\nClass balance (should be close to 50/50 by construction):")
print(targets.stack().value_counts(normalize=True))

Daily returns: (6511, 609)
Targets:       (6511, 609)

Class balance (should be close to 50/50 by construction):
1.0    0.50852
0.0    0.49148
Name: proportion, dtype: float64


## 3. Rolling study-period structure (paper Section 3.1)

In [6]:
# Cell 3
def generate_study_periods(dates: pd.DatetimeIndex, train_days: int = 750,
                            trade_days: int = 250) -> list[dict]:
    """Generate rolling study periods per Fischer & Krauss (2018) Section 3.1.
    Returns a list of dicts with integer index bounds and actual calendar
    dates for the training and trading/test windows of each period."""
    study_length = train_days + trade_days
    periods = []
    start_idx = 0
    period_num = 1

    while start_idx + study_length <= len(dates):
        train_start_idx = start_idx
        train_end_idx = start_idx + train_days - 1
        trade_start_idx = train_end_idx + 1
        trade_end_idx = start_idx + study_length - 1

        periods.append({
            "study_period": period_num,
            "train_start_idx": train_start_idx, "train_end_idx": train_end_idx,
            "trade_start_idx": trade_start_idx, "trade_end_idx": trade_end_idx,
            "train_start_date": dates[train_start_idx], "train_end_date": dates[train_end_idx],
            "trade_start_date": dates[trade_start_idx], "trade_end_date": dates[trade_end_idx],
        })

        start_idx += trade_days
        period_num += 1

    return periods


study_periods = generate_study_periods(price_panel.index)
study_period_index_df = pd.DataFrame(study_periods)
study_period_index_df.to_csv(PROCESSED_DIR / "study_period_index.csv", index=False)

print(f"Generated {len(study_periods)} study periods from the available date range.")
print(f"(Paper: 23 study periods from Dec 1989 -- Sep 2015; count here depends "
      f"on the actual date coverage of the downloaded data.)")

Generated 23 study periods from the available date range.
(Paper: 23 study periods from Dec 1989 -- Sep 2015; count here depends on the actual date coverage of the downloaded data.)


In [7]:
# Cell 4
# Explicit tabulation of the first 3 study periods, per the supervisor's request
display_cols = ["study_period", "train_start_date", "train_end_date",
                 "trade_start_date", "trade_end_date"]
first_three = study_period_index_df[display_cols].head(3).copy()
for col in ["train_start_date", "train_end_date", "trade_start_date", "trade_end_date"]:
    first_three[col] = first_three[col].dt.date

print("First 3 study periods (training windows overlap by 500 days.")
print("trading/test windows are back-to-back with no overlap and no gap):\n")
print(first_three.to_string(index=False))

First 3 study periods (training windows overlap by 500 days.
trading/test windows are back-to-back with no overlap and no gap):

 study_period train_start_date train_end_date trade_start_date trade_end_date
            1       1990-01-02     1992-12-16       1992-12-17     1993-12-13
            2       1990-12-27     1993-12-13       1993-12-14     1994-12-08
            3       1991-12-23     1994-12-08       1994-12-09     1995-12-05


## 4. Select the working study period

In [8]:
# Cell 5
STUDY_PERIOD_INDEX = 22

sp = study_periods[STUDY_PERIOD_INDEX]
train_dates = price_panel.index[sp["train_start_idx"]: sp["train_end_idx"] + 1]
trade_dates = price_panel.index[sp["trade_start_idx"]: sp["trade_end_idx"] + 1]

print(f"Working study period: {sp['study_period']}")
print(f"Training window: {train_dates[0].date()} to {train_dates[-1].date()} "
      f"({len(train_dates)} days)")
print(f"Trading/test window: {trade_dates[0].date()} to {trade_dates[-1].date()} "
      f"({len(trade_dates)} days)")

Working study period: 23
Training window: 2011-10-25 to 2014-10-17 (750 days)
Trading/test window: 2014-10-20 to 2015-10-15 (250 days)


## 6. LSTM input — standardized 240-day sequences

In [9]:
# Cell 6
SEQUENCE_LENGTH = 240
VALIDATION_FRACTION = 0.20  

def build_lstm_dataset(returns: pd.DataFrame, targets: pd.DataFrame,
                        train_dates: pd.DatetimeIndex, trade_dates: pd.DatetimeIndex,
                        sequence_length: int = SEQUENCE_LENGTH,
                        val_fraction: float = VALIDATION_FRACTION):
    """Build standardized 240-day sequences for the LSTM, split
    chronologically and leak-safely into train / validation / test."""

    n_val_days = int(len(train_dates) * val_fraction)
    chrono_train_dates = train_dates[:-n_val_days]
    chrono_val_dates = train_dates[-n_val_days:]


    train_returns_flat = returns.loc[chrono_train_dates].stack()
    mu = train_returns_flat.mean()
    sigma = train_returns_flat.std()

    standardized = (returns - mu) / sigma

    def extract_sequences(valid_dates: pd.DatetimeIndex):
        """Build one sequence per (date, ticker) where date is the TARGET
        date and must fall in valid_dates. The sequence's 240-day lookback
        window may reach earlier than valid_dates -- that is expected and
        is not leakage (see Section 5 markdown above); only past values are
        ever used, and only the target date's split membership matters."""
        X, y, meta = [], [], []
        for ticker in standardized.columns:
            series = standardized[ticker]
            tgt = targets[ticker]
            for date in valid_dates:
                if date not in series.index:
                    continue
                loc = series.index.get_loc(date)
                if loc < sequence_length - 1:
                    continue  # not enough history before this date yet
                window = series.iloc[loc - sequence_length + 1: loc + 1].values
                if np.isnan(window).any():
                    continue
                if pd.isna(tgt.loc[date]):
                    continue
                X.append(window)
                y.append(tgt.loc[date])
                meta.append((date, ticker))
        X = np.array(X).reshape(-1, sequence_length, 1) if X else np.empty((0, sequence_length, 1))
        y = np.array(y)
        return X, y, meta

    X_train, y_train, meta_train = extract_sequences(chrono_train_dates)
    X_val, y_val, meta_val = extract_sequences(chrono_val_dates)
    X_test, y_test, meta_test = extract_sequences(trade_dates)

    return {
        "train": (X_train, y_train, meta_train),
        "val": (X_val, y_val, meta_val),
        "test": (X_test, y_test, meta_test),
        "mu": mu, "sigma": sigma,
    }


lstm_data = build_lstm_dataset(daily_returns, targets, train_dates, trade_dates)

for split_name in ["train", "val", "test"]:
    X, y, meta = lstm_data[split_name]
    print(f"{split_name:5s}: X={X.shape}, y={y.shape}, "
          f"positive rate={y.mean():.3f}" if len(y) else f"{split_name:5s}: EMPTY")

train: X=(341257, 240, 1), y=(341257,), positive rate=0.501
val  : X=(87579, 240, 1), y=(87579,), positive rate=0.501
test : X=(148398, 240, 1), y=(148398,), positive rate=0.501


## 7. Benchmark input — 31 multi-period cumulative-return features

In [10]:
# Cell 7
CUMULATIVE_RETURN_HORIZONS = list(range(1, 21)) + list(range(40, 241, 20))
assert len(CUMULATIVE_RETURN_HORIZONS) == 31, "Expected 31 horizons per paper Section 3.4"

def build_benchmark_dataset(price_panel: pd.DataFrame, targets: pd.DataFrame,
                             train_dates: pd.DatetimeIndex, trade_dates: pd.DatetimeIndex,
                             val_fraction: float = VALIDATION_FRACTION):
    """Build the 31-feature cumulative-return table used by Logistic
    Regression, Random Forest, and DNN (paper Section 3.4), split
    chronologically and leak-safely."""

    n_val_days = int(len(train_dates) * val_fraction)
    chrono_train_dates = train_dates[:-n_val_days]
    chrono_val_dates = train_dates[-n_val_days:]

    feature_frames = []
    for m in CUMULATIVE_RETURN_HORIZONS:
        cum_ret = price_panel / price_panel.shift(m) - 1
        feature_frames.append(cum_ret.stack().rename(f"ret_{m}d"))

    features = pd.concat(feature_frames, axis=1)
    features.index.names = ["date", "ticker"]

    tgt_stacked = targets.stack()
    tgt_stacked.index.names = ["date", "ticker"]

    data = features.join(tgt_stacked.rename("target"), how="inner").dropna()

    def slice_split(valid_dates):
        return data[data.index.get_level_values("date").isin(valid_dates)]

    return {
        "train": slice_split(chrono_train_dates),
        "val": slice_split(chrono_val_dates),
        "test": slice_split(trade_dates),
    }


benchmark_data = build_benchmark_dataset(price_panel, targets, train_dates, trade_dates)

for split_name in ["train", "val", "test"]:
    df = benchmark_data[split_name]
    print(f"{split_name:5s}: {df.shape[0]} rows x {df.shape[1]} columns "
          f"(positive rate={df['target'].mean():.3f})" if len(df) else f"{split_name:5s}: EMPTY")


train: 341257 rows x 32 columns (positive rate=0.501)
val  : 87579 rows x 32 columns (positive rate=0.501)
test : 148398 rows x 32 columns (positive rate=0.501)


## 8. Save all outputs

In [11]:
# Cell 8
output_dir = PROCESSED_DIR / f"study_period_{STUDY_PERIOD_INDEX}"
output_dir.mkdir(parents=True, exist_ok=True)

for split_name in ["train", "val", "test"]:
    X, y, meta = lstm_data[split_name]
    dates_arr = np.array([m[0] for m in meta], dtype="datetime64[ns]") if meta else np.empty((0,))
    tickers_arr = np.array([m[1] for m in meta]) if meta else np.empty((0,))
    np.savez(
        output_dir / f"lstm_{split_name}.npz",
        X=X, y=y, dates=dates_arr, tickers=tickers_arr,
        mu=lstm_data["mu"], sigma=lstm_data["sigma"],
    )

for split_name in ["train", "val", "test"]:
    benchmark_data[split_name].to_csv(output_dir / f"benchmark_{split_name}.csv")

print(f"Saved all study period {STUDY_PERIOD_INDEX} outputs to: {output_dir.resolve()}")
print(f"\nFiles written:")
for f in sorted(output_dir.glob("*")):
    print(f"  {f.name}")

Saved all study period 22 outputs to: C:\Users\sahar\OneDrive\Desktop\Deep-Learning-Financial-Forecasting-main\data\processed\study_period_22

Files written:
  benchmark_test.csv
  benchmark_train.csv
  benchmark_val.csv
  lstm_test.npz
  lstm_train.npz
  lstm_val.npz


## 9. Sanity checks

Quick checks before handing off to the modeling notebooks: no NaNs leaked
through, class balance is reasonable in every split, and shapes are
internally consistent between the LSTM and benchmark datasets.


In [12]:
# Cell 9
for split_name in ["train", "val", "test"]:
    X, y, meta = lstm_data[split_name]
    if len(X) == 0:
        print(f"{split_name}: EMPTY -- check date ranges / sequence_length vs window size")
        continue
    print(f"{split_name}: any NaN in X? {np.isnan(X).any()} | "
          f"any NaN in y? {np.isnan(y).any()} | "
          f"class balance: {y.mean():.3f}")

print("\nBenchmark sanity checks")
for split_name in ["train", "val", "test"]:
    df = benchmark_data[split_name]
    if len(df) == 0:
        print(f"{split_name}: EMPTY")
        continue
    n_nan = df.isna().sum().sum()
    print(f"{split_name}: any NaN? {n_nan > 0} | class balance: {df['target'].mean():.3f}")

print("\nCross-check: no date overlap between splits")
train_d = set(d for d in lstm_data['train'][2] and [m[0] for m in lstm_data['train'][2]] or [])
val_d = set(d for d in lstm_data['val'][2] and [m[0] for m in lstm_data['val'][2]] or [])
test_d = set(d for d in lstm_data['test'][2] and [m[0] for m in lstm_data['test'][2]] or [])
print(f"train ∩ val dates:  {len(train_d & val_d)} (should be 0)")
print(f"train ∩ test dates: {len(train_d & test_d)} (should be 0)")
print(f"val ∩ test dates:   {len(val_d & test_d)} (should be 0)")

train: any NaN in X? False | any NaN in y? False | class balance: 0.501
val: any NaN in X? False | any NaN in y? False | class balance: 0.501
test: any NaN in X? False | any NaN in y? False | class balance: 0.501

Benchmark sanity checks
train: any NaN? False | class balance: 0.501
val: any NaN? False | class balance: 0.501
test: any NaN? False | class balance: 0.501

Cross-check: no date overlap between splits
train ∩ val dates:  0 (should be 0)
train ∩ test dates: 0 (should be 0)
val ∩ test dates:   0 (should be 0)
